# Transformer Dataset Prep and Validation Split - Christian De Guzman

This notebook turns the labeled contract chunks from `Labeling.ipynb` into a dataset the RoBERTa model can train on (Task #2). It:
1. loads the labeled chunks (`data/processed/*_chunks_labeled.parquet`) and the 41 categories,
2. moves about 15% of the **training contracts** into a validation set, so the test set is only used for final results,
3. builds the train / validation / test splits, with each chunk's 41 labels as a list of floats,
4. tokenizes each chunk with the `roberta-base` tokenizer (`input_ids` and `attention_mask`),
5. checks the splits, labels and tokens,
6. saves everything as a Hugging Face `DatasetDict` in `data/processed/roberta_dataset/`,
7. summarizes positives per category in each split.

**Why a validation set?** Training choices (epochs, class weights, decision thresholds) need to be checked on data the model hasn't trained on. If we used the test set for that, the final test scores would look better than they really are. The split is **by contract**, because neighboring chunks overlap by 128 tokens: splitting by chunk would put almost-identical text in both train and validation.

## 1. Setup

In [1]:
import json, os, shutil, tempfile
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict, Features, Sequence, Value, load_from_disk
from transformers import AutoTokenizer

C:\Users\CJ\accenture_1h\Accenture-1H-contract-review-challenge\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


All settings live in one place. Paths are relative to the `notebooks/` folder, like the other notebooks.

In [2]:
SEED = 42               # starting seed for the validation split
VAL_FRACTION = 0.15     # share of training contracts moved to validation
MAX_SPLIT_TRIES = 1000  # seeds to try before giving up on category coverage
MAX_LENGTH = 512        # roberta-base input limit
TOKENIZER_NAME = "roberta-base"

DATA_DIR = "../data/processed"
VAL_TITLES_PATH = f"{DATA_DIR}/val_titles.json"
DATASET_PATH = f"{DATA_DIR}/roberta_dataset"

if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(f"{DATA_DIR} not found. Run this notebook from the notebooks/ folder.")

## 2. Load the labeled chunks
These are the files saved by `Labeling.ipynb`. Each row is one chunk (up to 512 RoBERTa tokens, 128-token overlap) with `title`, `chunk_id`, `char_start`, `char_end`, `chunk_text`, then one 0/1 column for each of the 41 categories, in `categories.json` order.

In [3]:
train_df = pd.read_parquet(f"{DATA_DIR}/train_chunks_labeled.parquet")
test_df = pd.read_parquet(f"{DATA_DIR}/test_chunks_labeled.parquet")
with open(f"{DATA_DIR}/categories.json", "r") as file:
    CATEGORIES = json.load(file)

print("Train chunks:", len(train_df), "| Test chunks:", len(test_df), "| Categories:", len(CATEGORIES))

Train chunks: 14181 | Test chunks: 2898 | Categories: 41


In [4]:
assert len(CATEGORIES) == 41
assert list(train_df.columns[5:]) == CATEGORIES and list(test_df.columns[5:]) == CATEGORIES
assert train_df.shape == (14181, 46) and test_df.shape == (2898, 46)
assert train_df["title"].nunique() == 408 and test_df["title"].nunique() == 102
print("Load checks passed.")

Load checks passed.
